# NautilusTrader Orchestrator Tutorial

This notebook shows how to use `quant-orchestrator` with `quant-warehouse` for multi-vendor, single-symbol and multi-symbol research, Monte Carlo simulation, walk-forward optimization, equity-curve analysis, and portfolio optimization.


In [1]:

from __future__ import annotations

from pathlib import Path
import sys
from itertools import product
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.experiments import WindowSpec
from quant_orchestrator.monte_carlo import simulate_return_paths
from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.data_adapter import build_backtesting_frame
from quant_orchestrator.platforms.backtesting_frameworks.shared import MAG7_SYMBOLS, load_price_frame, load_signal_frame, normalize_session_label
from quant_orchestrator.strategy import summarize_backtest, summarize_equity
from quant_warehouse.feature_engineering import compute_features_worldclass

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)

SINGLE_SYMBOL = 'AAPL'
MULTI_VENDOR_PROVIDERS = ('fmp', 'yfinance')
TRAIN_START = '2020-01-01'
TRAIN_END = '2025-12-31'
TEST_START = '2026-01-01'
TEST_END = None
FAST_WINDOWS = (10, 20, 50)
SLOW_WINDOWS = (100, 150, 200)


def build_sma_frame(prices: pd.DataFrame, *, fast_window: int, slow_window: int) -> pd.DataFrame:
    if fast_window >= slow_window:
        raise ValueError('fast_window must be less than slow_window')

    frame = compute_features_worldclass(prices.copy())
    fast_col = f'SMA{fast_window}'
    slow_col = f'SMA{slow_window}'
    missing = [column for column in (fast_col, slow_col) if column not in frame.columns]
    if missing:
        raise ValueError(
            'Quant Warehouse feature output is missing required SMA columns: '
            f'{missing}. Update quant-warehouse feature engineering first.'
        )

    frame['fast_sma'] = frame[fast_col]
    frame['slow_sma'] = frame[slow_col]
    frame['signal'] = (frame['fast_sma'] > frame['slow_sma']).astype(int).fillna(0)
    frame = frame.dropna(subset=['open', 'high', 'low', 'close', 'volume']).copy()
    return frame



def combine_equity_sleeves(sleeves: list[pd.Series]) -> pd.Series:
    combined_index = pd.Index([])
    for sleeve in sleeves:
        combined_index = combined_index.union(sleeve.index)
    combined = pd.Series(0.0, index=combined_index, name='portfolio_value')
    for sleeve in sleeves:
        reindexed = sleeve.reindex(combined_index).ffill().fillna(sleeve.iloc[0])
        combined = combined.add(reindexed, fill_value=0.0)
    return combined.sort_index()


def align_return_frame(equity_map: dict[str, pd.Series]) -> pd.DataFrame:
    return pd.concat(
        {name: equity.pct_change().fillna(0.0) for name, equity in equity_map.items()},
        axis=1,
    ).fillna(0.0)


def max_sharpe_weights(return_frame: pd.DataFrame) -> pd.Series:
    if return_frame.empty:
        raise ValueError('return_frame cannot be empty')
    columns = list(return_frame.columns)
    mean = return_frame.mean().to_numpy(dtype=float)
    cov = return_frame.cov().to_numpy(dtype=float)
    if len(columns) == 1:
        return pd.Series([1.0], index=columns)

    def neg_sharpe(weights: np.ndarray) -> float:
        portfolio_mean = float(weights @ mean)
        portfolio_vol = float(np.sqrt(weights @ cov @ weights))
        if portfolio_vol <= 0:
            return 1e9
        return -(portfolio_mean / portfolio_vol)

    bounds = [(0.0, 1.0)] * len(columns)
    constraints = ({'type': 'eq', 'fun': lambda weights: float(np.sum(weights)) - 1.0},)
    guess = np.repeat(1.0 / len(columns), len(columns))
    result = minimize(neg_sharpe, guess, bounds=bounds, constraints=constraints, method='SLSQP')
    weights = pd.Series(result.x if result.success else guess, index=columns)
    weights = weights.clip(lower=0.0)
    return weights / weights.sum()


def weighted_equity_from_returns(return_frame: pd.DataFrame, weights: pd.Series, *, capital_base: float) -> pd.Series:
    aligned = return_frame.loc[:, weights.index].fillna(0.0)
    portfolio_returns = aligned.mul(weights, axis=1).sum(axis=1)
    equity = capital_base * (1.0 + portfolio_returns).cumprod()
    return equity.rename('portfolio_value')


def wfo_windows() -> WindowSpec:
    return WindowSpec(
        mode='fixed',
        train_start=TRAIN_START,
        train_end=TRAIN_END,
        test_start=TEST_START,
        test_end=TEST_END,
    )


def run_monte_carlo(equity: pd.Series, *, iterations: int = 1000, block_size: int = 5) -> pd.DataFrame:
    returns = equity.pct_change().dropna()
    mc = simulate_return_paths(returns, iterations=iterations, block_size=block_size, horizon=len(returns))
    return mc.summary


def summarize_candidates(rows: list[dict[str, object]]) -> pd.DataFrame:
    table = pd.DataFrame(rows)
    if table.empty:
        return table
    return table.sort_values(by=['total_return', 'max_drawdown', 'final_equity'], ascending=[False, True, False]).reset_index(drop=True)

FRAMEWORK_TITLE = 'NautilusTrader'


def run_framework_backtest(frame: pd.DataFrame, *, symbol: str, capital_base: float):
    from decimal import Decimal

    from nautilus_trader.backtest.engine import BacktestEngine
    from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, StrategyConfig
    from nautilus_trader.model.currencies import USD
    from nautilus_trader.model.enums import AccountType, OmsType, OrderSide, TimeInForce
    from nautilus_trader.model.objects import Money, Quantity
    from nautilus_trader.trading.strategy import Strategy
    from quant_orchestrator.platforms.backtesting_frameworks.nautilus.data_adapter import build_nautilus_in_memory_data

    adapter = build_nautilus_in_memory_data(frame, symbol=symbol, capital_base=capital_base)

    class SignalStrategyConfig(StrategyConfig, frozen=True):
        instrument_id: object
        bar_type: object
        trade_size: Decimal
        signal_map: dict

    class SignalStrategy(Strategy):
        def __init__(self, config: SignalStrategyConfig):
            super().__init__(config)
            self.is_long = False

        def on_start(self) -> None:
            self.subscribe_bars(self.config.bar_type)

        def on_bar(self, bar) -> None:
            bullish = self.config.signal_map.get(normalize_session_label(bar.ts_event), False)
            if bullish == self.is_long:
                return
            side = OrderSide.BUY if bullish else OrderSide.SELL
            order = self.order_factory.market(
                instrument_id=self.config.instrument_id,
                order_side=side,
                quantity=Quantity.from_int(int(self.config.trade_size)),
                time_in_force=TimeInForce.IOC,
            )
            self.submit_order(order)
            self.is_long = bullish

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            logging=LoggingConfig(log_level='OFF', bypass_logging=True),
        ),
    )
    engine.add_venue(
        venue=adapter.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.CASH,
        starting_balances=[Money(capital_base, USD)],
        base_currency=USD,
    )
    engine.add_instrument(adapter.instrument)
    engine.add_data(adapter.bars)
    engine.add_strategy(
        SignalStrategy(
            SignalStrategyConfig(
                instrument_id=adapter.instrument.id,
                bar_type=adapter.bar_type,
                trade_size=Decimal(adapter.trade_size),
                signal_map=adapter.signal_map,
            ),
        ),
    )

    started = perf_counter()
    engine.run()
    fills_report = engine.trader.generate_order_fills_report()
    elapsed = perf_counter() - started
    engine.dispose()

    cash = float(capital_base)
    position = 0.0
    values = []
    fills_by_date: dict[pd.Timestamp, list[pd.Series]] = {}
    for _, fill in fills_report.iterrows():
        fills_by_date.setdefault(normalize_session_label(fill['ts_last']), []).append(fill)
    for date, row in frame.iterrows():
        normalized = normalize_session_label(date)
        for fill in fills_by_date.get(normalized, []):
            quantity = float(fill['filled_qty'])
            price = float(fill['avg_px'])
            if str(fill['side']) == 'BUY':
                cash -= quantity * price
                position += quantity
            else:
                cash += quantity * price
                position -= quantity
        values.append(cash + position * float(row['close']))
    equity = pd.Series(values, index=frame.index, name='portfolio_value')
    summary = summarize_backtest(
        framework='nautilus',
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(frame),
        trades=len(fills_report),
    )
    summary['native_fills'] = int(len(fills_report))
    summary['native_last_value'] = float(equity.iloc[-1])
    return fills_report, summary, equity


## Single Symbol / Multi Vendor


In [2]:

print(f'Tutorial framework: {FRAMEWORK_TITLE}')
print(f'Single symbol: {SINGLE_SYMBOL}')
print(f'Vendors: {MULTI_VENDOR_PROVIDERS}')


Tutorial framework: NautilusTrader
Single symbol: AAPL
Vendors: ('fmp', 'yfinance')


## Multi Vendor Backtest


In [3]:

def run_multi_vendor_single_symbol(symbol: str = SINGLE_SYMBOL, providers: tuple[str, ...] = MULTI_VENDOR_PROVIDERS, capital_base: float = 100_000.0) -> pd.DataFrame:
    rows = []
    for provider in providers:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        rows.append(summary.assign(provider=provider))
    return pd.concat(rows, ignore_index=True)

vendor_report = run_multi_vendor_single_symbol()
display(vendor_report.round(4).head(10))


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second,native_fills,native_last_value,provider
0,nautilus,AAPL,2020-10-15,2026-06-24,1428,9,111919.48,0.1192,-0.1410,0.0052,0.0470,30397.22,9,111919.4800,fmp
1,nautilus,AAPL,2020-10-15,2026-06-23,1427,9,110360.35,0.1036,-0.1402,0.0051,0.0384,37173.53,9,110360.3475,yfinance


## Multi Symbol Backtest


In [4]:

def run_multi_symbol_backtest(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    symbol_rows = []
    sleeves = []
    runs = {}
    elapsed = 0.0
    trades = 0
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        started = perf_counter()
        raw, summary, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        elapsed += perf_counter() - started
        symbol_rows.append(summary.assign(provider=provider))
        sleeves.append(equity.rename(symbol))
        trades += int(summary['trades'].iloc[0])
        runs[symbol] = raw
    portfolio_equity = combine_equity_sleeves(sleeves)
    portfolio_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol='MAG7',
        equity=portfolio_equity,
        elapsed_seconds=elapsed,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary['provider'] = provider
    return pd.concat(symbol_rows, ignore_index=True), portfolio_summary, runs, portfolio_equity

symbol_report, portfolio_report, runs_by_symbol, portfolio_equity = run_multi_symbol_backtest()
print('Symbol report:')
display(symbol_report.loc[:, ['provider', 'symbol', 'start', 'end', 'bars', 'trades', 'final_equity', 'total_return', 'max_drawdown', 'elapsed_seconds']].round(4).head(10))
print()
print('Portfolio report:')
display(portfolio_report.round(4).head(10))


Symbol report:


,provider,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,fmp,AAPL,2020-10-15,2026-06-24,1428,9,15964.51,0.1175,-0.1393,0.0391
1,fmp,AMZN,2020-10-15,2026-06-24,1428,9,13734.04,-0.0386,-0.1529,0.0405
2,fmp,GOOGL,2020-10-15,2026-06-24,1428,5,25448.41,0.7814,-0.1365,0.0366
3,fmp,META,2020-10-15,2026-06-24,1428,6,19795.37,0.3857,-0.1522,0.0384
4,fmp,MSFT,2020-10-15,2026-06-24,1428,10,16229.55,0.1361,-0.1211,0.0785
5,fmp,NVDA,2020-10-15,2026-06-24,1428,5,52030.81,2.6422,-0.2192,0.0366
6,fmp,TSLA,2020-10-15,2026-06-24,1428,12,13046.93,-0.0867,-0.3453,0.0410



Portfolio report:


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second,provider
0,NautilusTrader,MAG7,2020-10-15,2026-06-24,1428,56,156249.65,0.5625,-0.1241,0.0066,0.4541,3144.72,fmp


## Monte Carlo


In [5]:

mc_report = run_monte_carlo(portfolio_equity, iterations=1000, block_size=5)
display(mc_report.round(4).head(10))


,iterations,horizon,terminal_return_mean,terminal_return_p05,terminal_return_p50,terminal_return_p95,max_drawdown_mean,max_drawdown_p05
0,1000,1427,0.5937,0.0774,0.5516,1.245,-0.1501,-0.2411


## Walk Forward Optimization


In [6]:

def run_walk_forward_optimization(symbol: str = SINGLE_SYMBOL, provider: str = 'fmp', capital_base: float = 100_000.0):
    spec = wfo_windows()
    train_frame = load_price_frame(symbol, provider=provider, start=spec.train_start, end=spec.train_end)
    test_frame = load_price_frame(symbol, provider=provider, start=spec.test_start, end=spec.test_end)

    train_rows = []
    for fast_window, slow_window in product(FAST_WINDOWS, SLOW_WINDOWS):
        if fast_window >= slow_window:
            continue
        frame = build_sma_frame(train_frame, fast_window=fast_window, slow_window=slow_window)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        row = summary.iloc[0].to_dict()
        row['fast_window'] = fast_window
        row['slow_window'] = slow_window
        train_rows.append(row)

    train_grid = summarize_candidates(train_rows)
    best = train_grid.iloc[0]
    best_fast = int(best['fast_window'])
    best_slow = int(best['slow_window'])
    _, _, test_equity = run_framework_backtest(build_sma_frame(test_frame, fast_window=best_fast, slow_window=best_slow), symbol=symbol, capital_base=capital_base)
    test_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol=symbol,
        equity=test_equity,
        elapsed_seconds=0.0,
        bars=len(test_frame),
        trades=None,
    )
    return train_grid, best, test_summary

train_grid, best_row, test_summary = run_walk_forward_optimization()
print('Train grid:')
display(train_grid.loc[:, ['fast_window', 'slow_window', 'final_equity', 'total_return', 'max_drawdown']].round(4).head(10))
print()
print('Best parameters:')
display(best_row.loc[['fast_window', 'slow_window', 'total_return', 'max_drawdown']].to_frame(name='value').round(4))
print()
print('Test summary:')
display(test_summary.round(4))


Train grid:


,fast_window,slow_window,final_equity,total_return,max_drawdown
0,50,100,143235.40,0.4324,-0.1282
1,10,200,140875.60,0.4088,-0.1341
2,20,100,140513.35,0.4051,-0.1842
3,10,100,139236.85,0.3924,-0.1637
4,20,200,131329.45,0.3133,-0.1969
5,10,150,125954.35,0.2595,-0.1733
6,20,150,125564.50,0.2556,-0.1425
7,50,150,124049.95,0.2405,-0.1473
8,50,200,111809.35,0.1181,-0.2102



Best parameters:


,value
fast_window,50
slow_window,100
total_return,0.4324
max_drawdown,-0.1282



Test summary:


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second
0,NautilusTrader,AAPL,2026-01-02,2026-06-24,119,None,98370.68,-0.0163,-0.0235,0.0016,0.0,None


## Equity Curve Analysis


In [7]:

summary_view = pd.Series(summarize_equity(portfolio_equity), name='value').to_frame()
display(summary_view.round(4))
display(portfolio_equity.tail().to_frame(name='portfolio_value').round(2))


,value
start,2020-10-15
end,2026-06-24
final_equity,156249.65
total_return,0.5625
max_drawdown,-0.1241
daily_vol,0.0066


,portfolio_value
2026-06-17 00:00:00,158676.83
2026-06-18 00:00:00,160614.32
2026-06-22 00:00:00,158994.77
2026-06-23 00:00:00,156585.89
2026-06-24 00:00:00,156249.65


## Portfolio Optimization


In [8]:

def optimize_symbol_portfolio(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    equity_map = {}
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, _, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        equity_map[symbol] = equity
    return equity_map

symbol_equities = optimize_symbol_portfolio()
symbol_returns = align_return_frame(symbol_equities)
symbol_weights = max_sharpe_weights(symbol_returns)
symbol_portfolio_equity = weighted_equity_from_returns(symbol_returns, symbol_weights, capital_base=100_000.0)
print('Symbol-level weights:')
display(symbol_weights.sort_values(ascending=False).to_frame(name='weight').round(4))
print()
print('Symbol-level portfolio summary:')
display(pd.Series(summarize_equity(symbol_portfolio_equity), name='value').to_frame().round(4))

candidate_pairs = [(fast, slow) for fast in FAST_WINDOWS for slow in SLOW_WINDOWS if fast < slow]
strategy_equities = {}
strategy_rows = []
strategy_frame = load_price_frame(SINGLE_SYMBOL, provider='fmp', start=TRAIN_START, end=TEST_END)
for fast_window, slow_window in candidate_pairs:
    candidate_frame = build_sma_frame(strategy_frame, fast_window=fast_window, slow_window=slow_window)
    _, summary, equity = run_framework_backtest(candidate_frame, symbol=SINGLE_SYMBOL, capital_base=100_000.0)
    key = f'{fast_window}/{slow_window}'
    strategy_equities[key] = equity
    strategy_rows.append({
        'strategy': key,
        'fast_window': fast_window,
        'slow_window': slow_window,
        'total_return': float(summary['total_return'].iloc[0]),
        'max_drawdown': float(summary['max_drawdown'].iloc[0]),
    })
strategy_returns = align_return_frame(strategy_equities)
strategy_weights = max_sharpe_weights(strategy_returns)
strategy_portfolio_equity = weighted_equity_from_returns(strategy_returns, strategy_weights, capital_base=100_000.0)
print()
print('Strategy-level weights:')
display(strategy_weights.sort_values(ascending=False).head(10).to_frame(name='weight').round(4))
print()
print('Strategy-level portfolio summary:')
display(pd.Series(summarize_equity(strategy_portfolio_equity), name='value').to_frame().round(4))


Symbol-level weights:


,weight
GOOGL,0.5652
META,0.2267
NVDA,0.2081
MSFT,0.0000
AMZN,0.0000
AAPL,0.0000
TSLA,0.0000



Symbol-level portfolio summary:


,value
start,2020-10-15
end,2026-06-24
final_equity,201889.12
total_return,1.0189
max_drawdown,-0.1135
daily_vol,0.0064



Strategy-level weights:


,weight
10/200,0.6871
50/100,0.3116
10/100,0.0014
10/150,0.0000
20/150,0.0000
20/200,0.0000
50/150,0.0000
20/100,0.0000
50/200,0.0000



Strategy-level portfolio summary:


,value
start,2020-01-02
end,2026-06-24
final_equity,146034.17
total_return,0.4603
max_drawdown,-0.1192
daily_vol,0.0062
